In [2]:
import tensorflow as tf
import re
import datetime
import importlib # To dynamically get model functions

# Get current time and location context
try:
    # Use zoneinfo if available (Python 3.9+)
    from zoneinfo import ZoneInfo
    tz = ZoneInfo("Europe/Helsinki") # EEST corresponds to Europe/Helsinki
except ImportError:
    # Fallback for older Python versions (requires pytz installed: pip install pytz)
    try:
        import pytz
        tz = pytz.timezone("Europe/Helsinki")
    except ImportError:
        # Fallback to UTC if no timezone library found
        print("Warning: Timezone libraries (zoneinfo/pytz) not found. Using UTC.")
        tz = datetime.timezone.utc

now = datetime.datetime.now(tz)
location = "Oulu, North Ostrobothnia, Finland"
print(f"Code execution time: {now.strftime('%Y-%m-%d %H:%M:%S %Z')}")
print(f"Current location context: {location}")


def find_block_start_layers(model_name, model_fn, input_shape=(224, 224, 3)):
    """
    Loads a Keras application model and identifies the starting layer index and name
    for each major block/stage, using model-specific naming conventions.

    Args:
        model_name (str): A descriptive name for the model (for printing).
        model_fn (callable): The Keras function to load the model
                             (e.g., tf.keras.applications.ResNet50).
        input_shape (tuple): The input shape for the model.

    Returns:
        dict: A dictionary where keys are block/stage identifiers
              (e.g., 'conv2_block1', 'block3', 'mixed5', 'stage1')
              and values are tuples (layer_index, layer_name).
              Returns None if the model fails to load.
    """
    print(f"\n--- Analyzing Model: {model_name} ---")
    try:
        # Load the base model without the classification head
        model = model_fn(include_top=False, weights='imagenet', input_shape=input_shape)
        # Use the actual name Keras assigns to the model
        model_actual_name = model.name
        print(f"Successfully loaded '{model_actual_name}' with {len(model.layers)} layers.")
    except Exception as e:
        print(f"Error loading model {model_name} (function: {model_fn.__name__}): {e}")
        return None

    block_starts = {}

    # --- Define Patterns (using common Keras naming) ---
    # NOTE: These patterns might need minor adjustments based on specific TF versions

    # --- Define Patterns (using common Keras naming) ---
    # Add VGG16 pattern
    vgg_pattern = re.compile(r'^block(\d+)_conv1$')  # Matches blockX_conv1

    # ResNet V1 (e.g., ResNet50): conv<stage>_block1... marks stage start
    # Residual connections often 'res<stage>...'
    resnet_v1_pattern = re.compile(r'^conv(\d+)_block1_')

    # MobileNetV2: block_<block_num>_<op>... Start often marked by 'expand' or 'depthwise'
    mobilenet_v2_pattern = re.compile(r'^block_(\d+)_(expand|depthwise)')

    # InceptionV3: Modules named 'mixed<num>'
    inception_v3_pattern = re.compile(r'^(mixed\d+)') # Capture the whole 'mixedX' name

    # EfficientNet (V1/V2): block<num>[a-z]... Start marked by block<num>a...
    # V1 (B0-B7), V2 (B0-B3, S, M, L) use similar block naming for start detection
    efficientnet_pattern = re.compile(r'^block(\d+)[a-z]_')

    # ConvNeXt: <model_name>_stage_<num>_block_<num>_depthwise_conv
    # We need model_actual_name AFTER loading
    convnext_pattern = None # Will be defined after model load if it's a ConvNeXt model
    if 'convnext' in model_actual_name:
         convnext_pattern_str = rf"^{model_actual_name}_stage_(\d+)_block_(\d+)_depthwise_conv$"
         convnext_pattern = re.compile(convnext_pattern_str)
         print(f"Using ConvNeXt pattern: {convnext_pattern_str}")


    print("Searching for block/stage start patterns...")
    identified_model_type = "Unknown"

    for i, layer in enumerate(model.layers):
        layer_name = layer.name
        block_found_for_layer = False # Flag to process only the first matching pattern per layer

        # --- Apply Pattern Based on Loaded Model Name ---
        
        # Add VGG16 Check before other patterns
        if 'vgg16' in model_actual_name and not block_found_for_layer:
            identified_model_type = "VGG16"
            match = vgg_pattern.match(layer_name)
            if match:
                block_num = int(match.group(1))
                block_id = f'block{block_num}'
                if block_id not in block_starts:
                    print(f"  Found VGG16 {block_id} start: Layer {i} ('{layer_name}')")
                    block_starts[block_id] = (i, layer_name)
                    block_found_for_layer = True

        # ConvNeXt Check (most specific name structure)
        if convnext_pattern and not block_found_for_layer:
            identified_model_type = "ConvNeXt"
            match = convnext_pattern.match(layer_name)
            if match:
                stage_num = int(match.group(1))
                block_id = f'stage{stage_num}' # Stage number is the key identifier
                if block_id not in block_starts:
                    print(f"  Found ConvNeXt {block_id} start: Layer {i} ('{layer_name}')")
                    block_starts[block_id] = (i, layer_name)
                    block_found_for_layer = True

        # EfficientNetV2 Check (contains 'efficientnetv2')
        if 'efficientnetv2' in model_actual_name and not block_found_for_layer:
            identified_model_type = "EfficientNetV2"
            match = efficientnet_pattern.match(layer_name)
            if match:
                block_num = int(match.group(1))
                block_id = f'block{block_num}'
                if block_id not in block_starts:
                    print(f"  Found EfficientNetV2 {block_id} start: Layer {i} ('{layer_name}')")
                    block_starts[block_id] = (i, layer_name)
                    block_found_for_layer = True

        # EfficientNet V1 Check (contains 'efficientnet', but not 'v2')
        elif 'efficientnet' in model_actual_name and not block_found_for_layer:
            identified_model_type = "EfficientNetV1"
            match = efficientnet_pattern.match(layer_name)
            if match:
                block_num = int(match.group(1))
                block_id = f'block{block_num}'
                if block_id not in block_starts:
                    print(f"  Found EfficientNetV1 {block_id} start: Layer {i} ('{layer_name}')")
                    block_starts[block_id] = (i, layer_name)
                    block_found_for_layer = True

        # ResNet50 V1 Check (specific name 'resnet50')
        elif model_actual_name == 'resnet50' and not block_found_for_layer:
            identified_model_type = "ResNet50V1"
            match = resnet_v1_pattern.match(layer_name)
            if match:
                block_num = int(match.group(1))
                # Use convX as the identifier for ResNet stages/blocks
                block_id = f'conv{block_num}'
                if block_num >= 2 and block_id not in block_starts:
                    print(f"  Found ResNet50 {block_id} start: Layer {i} ('{layer_name}')")
                    block_starts[block_id] = (i, layer_name)
                    block_found_for_layer = True

        # MobileNetV2 Check
        elif 'mobilenetv2' in model_actual_name and not block_found_for_layer:
            identified_model_type = "MobileNetV2"
            match = mobilenet_v2_pattern.match(layer_name)
            if match:
                block_num = int(match.group(1))
                block_id = f'block_{block_num}' # Use block_X identifier
                if block_id not in block_starts:
                     print(f"  Found MobileNetV2 {block_id} start: Layer {i} ('{layer_name}')")
                     block_starts[block_id] = (i, layer_name)
                     block_found_for_layer = True

        # InceptionV3 Check
        elif 'inception_v3' in model_actual_name and not block_found_for_layer:
             identified_model_type = "InceptionV3"
             match = inception_v3_pattern.match(layer_name)
             if match:
                 block_id = match.group(1) # block_id is 'mixedX'
                 if block_id not in block_starts:
                     print(f"  Found InceptionV3 {block_id} start: Layer {i} ('{layer_name}')")
                     block_starts[block_id] = (i, layer_name)
                     block_found_for_layer = True

        # Add elif clauses here for other models if needed (e.g., ResNetV2 variants)


    # Add Input layer and attempt to find stem start
    if model.layers:
         block_starts['input_layer'] = (0, model.layers[0].name)
         # Find first conv/stem layer (can vary significantly)
         stem_found = False
         # Start searching after input layer
         for i, layer in enumerate(model.layers[1:], start=1):
             is_conv = isinstance(layer, (tf.keras.layers.Conv2D, tf.keras.layers.DepthwiseConv2D))
             # ConvNeXt uses Patchify/Embedding, Inception/MobileNet/ResNet start with Conv2D
             is_patch_or_embed = 'patch' in layer.name.lower() or 'embedding' in layer.name.lower()

             # Generic check for first Conv or Patch layer after input
             if (is_conv or is_patch_or_embed) and 'stem_start' not in block_starts:
                 block_starts['stem_start'] = (i, layer.name)
                 print(f"  Found stem start ({identified_model_type}): Layer {i} ('{layer.name}')")
                 stem_found = True
                 break # Found the first one

         if not stem_found and len(model.layers) > 1: # Fallback if specific stem not found
             block_starts['stem_start'] = (1, model.layers[1].name) # Assume layer 1 starts processing
             print(f"  Found stem start (fallback layer 1): Layer 1 ('{model.layers[1].name}')")


    print(f"--- Analysis Complete for {model_actual_name} ({identified_model_type}) ---")
    return block_starts

# --- Models to Analyze ---
models_to_analyze = {
    # Name: Keras Application Function
    'ResNet50': tf.keras.applications.ResNet50,
    # 'MobileNetV2': tf.keras.applications.MobileNetV2,
    # 'InceptionV3': tf.keras.applications.InceptionV3,
    # 'EfficientNetB0': tf.keras.applications.EfficientNetB0, # V1
    # 'EfficientNetV2B0': tf.keras.applications.EfficientNetV2B0,
    # 'EfficientNetV2B1': tf.keras.applications.EfficientNetV2B1,
    # 'ConvNeXtTiny': tf.keras.applications.ConvNeXtTiny,
    # 'ConvNeXtSmall': tf.keras.applications.ConvNeXtSmall,
    # 'ConvNeXtBase': tf.keras.applications.ConvNeXtBase,
    # 'ConvNeXtLarge': tf.keras.applications.ConvNeXtLarge,
    # Add other models here if needed, e.g., ResNet50V2
    # 'ResNet50V2': tf.keras.applications.ResNet50V2,
    'VGG16': tf.keras.applications.VGG16,
}

# --- Run Analysis for All Models ---
all_block_starts = {}
INPUT_SHAPE = (224, 224, 3) # Common input shape, adjust if needed

for name, model_func in models_to_analyze.items():
    block_starts = find_block_start_layers(name, model_func, INPUT_SHAPE)
    if block_starts:
        all_block_starts[name] = block_starts
        # Print summary for verification
        print(f"  Summary for {name}:")
        for block_id, (idx, layer_name) in block_starts.items():
             print(f"    - {block_id:<15}: Index {idx:<4}, Name '{layer_name}'")
    else:
        print(f"  Skipping summary for {name} due to loading error.")


# --- Helper Function for Unfreezing (from previous example) ---
# def unfreeze_from_block(model, block_starts, block_to_unfreeze_name, verbose=True):
#     """Unfreezes layers from a specific block/stage onwards."""
#     model_actual_name = model.name # Use actual name for messages
#     if not block_starts:
#         print(f"Error: block_starts dictionary is empty or None for {model_actual_name}.")
#         return
#     if block_to_unfreeze_name not in block_starts:
#         print(f"\nError: Block/Stage name '{block_to_unfreeze_name}' not found in block_starts for {model_actual_name}.")
#         print(f"Available identifiers: {list(block_starts.keys())}")
#         return False # Indicate failure

#     start_index, start_name = block_starts[block_to_unfreeze_name]
#     if verbose:
#         print(f"\nUnfreezing layers in {model_actual_name} from '{block_to_unfreeze_name}' (Layer index {start_index}, name '{start_name}') onwards...")

#     unfrozen_count = 0
#     # Iterate from the identified start index to the end
#     for layer in model.layers[start_index:]:
#         # Only set trainable if the layer has trainable weights
#         if len(layer.trainable_weights) > 0:
#              layer.trainable = True
#              unfrozen_count += 1

#     if verbose:
#         print(f"Unfreezing complete for {model_actual_name}. Set {unfrozen_count} layers from index {start_index} onwards to trainable.")
#         print("Remember to recompile the model if needed after changing trainable status.")
#     return True # Indicate success


# --- Example: Fine-tune ConvNeXtBase from stage3 ---
# print("\n--- Fine-tuning Example ---")
# try:
#     # 1. Load the model
#     model_cnb = tf.keras.applications.ConvNeXtBase(include_top=False, weights='imagenet', input_shape=INPUT_SHAPE)

#     # 2. Freeze the entire model initially
#     print(f"\nFreezing all layers in {model_cnb.name} initially...")
#     model_cnb.trainable = False
#     print("Model frozen.")

#     # 3. Selectively unfreeze using the results from our analysis
#     if 'ConvNeXtBase' in all_block_starts:
#         success = unfreeze_from_block(model_cnb, all_block_starts['ConvNeXtBase'], 'stage3')
#         if success:
#              # 4. Recompile the model with changes in trainable status
#              # Use a low learning rate for fine-tuning
#              print("\nRecompiling model with low learning rate...")
#              optimizer = tf.keras.optimizers.Adam(learning_rate=1e-5)
#              model_cnb.compile(optimizer=optimizer, loss='...', metrics=['...']) # Add your loss and metrics
#              print("Model recompiled.")
#              # model_cnb.summary() # Optional: Check trainable params
#     else:
#         print("Block starts for ConvNeXtBase not found, skipping fine-tuning steps.")

# except Exception as e:
#     print(f"\nError during ConvNeXtBase fine-tuning example: {e}")

Code execution time: 2025-04-10 15:34:05 EEST
Current location context: Oulu, North Ostrobothnia, Finland

--- Analyzing Model: ResNet50 ---
Successfully loaded 'resnet50' with 175 layers.
Searching for block/stage start patterns...
  Found ResNet50 conv2 start: Layer 7 ('conv2_block1_1_conv')
  Found ResNet50 conv3 start: Layer 39 ('conv3_block1_1_conv')
  Found ResNet50 conv4 start: Layer 81 ('conv4_block1_1_conv')
  Found ResNet50 conv5 start: Layer 143 ('conv5_block1_1_conv')
  Found stem start (ResNet50V1): Layer 2 ('conv1_conv')
--- Analysis Complete for resnet50 (ResNet50V1) ---
  Summary for ResNet50:
    - conv2          : Index 7   , Name 'conv2_block1_1_conv'
    - conv3          : Index 39  , Name 'conv3_block1_1_conv'
    - conv4          : Index 81  , Name 'conv4_block1_1_conv'
    - conv5          : Index 143 , Name 'conv5_block1_1_conv'
    - input_layer    : Index 0   , Name 'input_4'
    - stem_start     : Index 2   , Name 'conv1_conv'

--- Analyzing Model: VGG16 ---
